# Step 9 - Lag Selection and Candidate Orders

This notebook converts the ACF and PACF diagnostics into a shortlist of concrete SARIMA orders. The aim is to keep the candidate set compact, defensible, and easy to compare with AIC and BIC in the next step.


In [1]:
import os

import pandas as pd
from IPython.display import display

PROCESSED_PATH = "../data/processed/"

acf_pacf_summary = pd.read_csv(os.path.join(PROCESSED_PATH, "acf_pacf_summary.csv"))
model_identification_summary = pd.read_csv(
    os.path.join(PROCESSED_PATH, "model_identification_summary.csv")
)

display(model_identification_summary)
display(acf_pacf_summary.head(30))


,selected_model_family,selected_order_template,selected_seasonal_order_template,primary_seasonal_period,secondary_cycle_hours,recommended_train_window_hours,recommended_validation_window_hours,reason
0,SARIMA,"(p,1,q)","(P,1,Q,24)",24,168,3984,336,The raw series is non-stationary and strongly ...


,lag,acf,pacf,significant_acf,significant_pacf,suggested_signal
0,1,0.763622,0.763622,True,True,shared_acf_pacf_signal
1,2,0.413442,-0.407012,True,True,shared_acf_pacf_signal
2,3,0.166662,0.095602,True,True,shared_acf_pacf_signal
3,4,0.031506,-0.047310,True,True,shared_acf_pacf_signal
4,5,-0.049290,-0.060238,True,True,shared_acf_pacf_signal
5,6,-0.100171,-0.030082,True,True,shared_acf_pacf_signal
6,7,-0.123119,-0.023134,True,False,possible_ma_signal
7,8,-0.118035,-0.005967,True,False,possible_ma_signal
8,9,-0.067182,0.074887,True,True,shared_acf_pacf_signal
9,10,-0.017596,-0.042185,False,True,possible_ar_signal


In [2]:
short_lag_view = acf_pacf_summary.loc[acf_pacf_summary["lag"].between(1, 6)].copy()
seasonal_lag_view = acf_pacf_summary.loc[acf_pacf_summary["lag"].isin([24, 25, 48])].copy()

print("Short-run lag diagnostics")
display(short_lag_view.style.format({"acf": "{:,.4f}", "pacf": "{:,.4f}"}))

print("Seasonal lag diagnostics")
display(seasonal_lag_view.style.format({"acf": "{:,.4f}", "pacf": "{:,.4f}"}))


Short-run lag diagnostics


,lag,acf,pacf,significant_acf,significant_pacf,suggested_signal
0,1,0.7636,0.7636,True,True,shared_acf_pacf_signal
1,2,0.4134,-0.4070,True,True,shared_acf_pacf_signal
2,3,0.1667,0.0956,True,True,shared_acf_pacf_signal
3,4,0.0315,-0.0473,True,True,shared_acf_pacf_signal
4,5,-0.0493,-0.0602,True,True,shared_acf_pacf_signal
5,6,-0.1002,-0.0301,True,True,shared_acf_pacf_signal


Seasonal lag diagnostics


,lag,acf,pacf,significant_acf,significant_pacf,suggested_signal
23,24,-0.1781,-0.0717,True,True,shared_acf_pacf_signal
24,25,-0.1434,0.1447,True,True,shared_acf_pacf_signal
47,48,-0.3194,-0.0127,True,False,possible_ma_signal


The shortlist should stay narrow on purpose. Once the series has already been differenced and seasonally differenced, very large AR and MA orders are usually unnecessary and can make the fitting step unstable without adding much forecasting value.


In [3]:
sarima_candidates = pd.DataFrame(
    [
        {
            "label": "arima_baseline_211",
            "p": 2,
            "d": 1,
            "q": 1,
            "P": 0,
            "D": 0,
            "Q": 0,
            "s": 0,
            "reason": "Non-seasonal benchmark using the strongest short-run lags.",
        },
        {
            "label": "sarima_111_011_24",
            "p": 1,
            "d": 1,
            "q": 1,
            "P": 0,
            "D": 1,
            "Q": 1,
            "s": 24,
            "reason": "Parsimonious daily seasonal benchmark.",
        },
        {
            "label": "sarima_211_011_24",
            "p": 2,
            "d": 1,
            "q": 1,
            "P": 0,
            "D": 1,
            "Q": 1,
            "s": 24,
            "reason": "Adds one more non-seasonal AR lag suggested by the PACF.",
        },
        {
            "label": "sarima_121_011_24",
            "p": 1,
            "d": 1,
            "q": 2,
            "P": 0,
            "D": 1,
            "Q": 1,
            "s": 24,
            "reason": "Tests whether a second MA lag improves the short-run error structure.",
        },
        {
            "label": "sarima_111_111_24",
            "p": 1,
            "d": 1,
            "q": 1,
            "P": 1,
            "D": 1,
            "Q": 1,
            "s": 24,
            "reason": "Adds a seasonal AR term in case lag-24 persistence remains meaningful.",
        },
        {
            "label": "sarima_211_111_24",
            "p": 2,
            "d": 1,
            "q": 1,
            "P": 1,
            "D": 1,
            "Q": 1,
            "s": 24,
            "reason": "Full low-order daily SARIMA candidate with both AR dimensions active.",
        },
    ]
)

display(sarima_candidates)


,label,p,d,q,P,D,Q,s,reason
0,arima_baseline_211,2,1,1,0,0,0,0,Non-seasonal benchmark using the strongest sho...
1,sarima_111_011_24,1,1,1,0,1,1,24,Parsimonious daily seasonal benchmark.
2,sarima_211_011_24,2,1,1,0,1,1,24,Adds one more non-seasonal AR lag suggested by...
3,sarima_121_011_24,1,1,2,0,1,1,24,Tests whether a second MA lag improves the sho...
4,sarima_111_111_24,1,1,1,1,1,1,24,Adds a seasonal AR term in case lag-24 persist...
5,sarima_211_111_24,2,1,1,1,1,1,24,Full low-order daily SARIMA candidate with bot...


In [4]:
sarima_candidates.to_csv(
    os.path.join(PROCESSED_PATH, "sarima_candidate_orders.csv"), index=False
)

print("Saved candidate model list to ../data/processed/sarima_candidate_orders.csv")
sarima_candidates


Saved candidate model list to ../data/processed/sarima_candidate_orders.csv


,label,p,d,q,P,D,Q,s,reason
0,arima_baseline_211,2,1,1,0,0,0,0,Non-seasonal benchmark using the strongest sho...
1,sarima_111_011_24,1,1,1,0,1,1,24,Parsimonious daily seasonal benchmark.
2,sarima_211_011_24,2,1,1,0,1,1,24,Adds one more non-seasonal AR lag suggested by...
3,sarima_121_011_24,1,1,2,0,1,1,24,Tests whether a second MA lag improves the sho...
4,sarima_111_111_24,1,1,1,1,1,1,24,Adds a seasonal AR term in case lag-24 persist...
5,sarima_211_111_24,2,1,1,1,1,1,24,Full low-order daily SARIMA candidate with bot...


Final shortlist: keep one non-seasonal benchmark and a handful of low-order daily SARIMA models. That gives the next notebook enough variety to compare fit quality without turning the project into an unbounded grid search.
